# CCDC → CSV → CIF → PDF → Markdown → 结构卡片

本 Notebook 是按需执行的工作面板，不要使用 Run All。先核对每个单元的输入和预览，再运行需要的步骤；下载、MinerU 和模型步骤默认关闭执行开关。（readme.md有各cell详细信息）

- 原始 `.gcd`：`inputs/`
- 中间 CSV：`data/`
- CIF：`CIFs/`
- PDF：`documents/pdf/`
- MinerU Markdown：`documents/md/`
- 结构卡片：`data/extracted_cards/` 与 `data/cif_structure_cards/`

先运行初始化单元。检查环境情况。项目根目录从当前目录祖先自动发现，也可设置 `CLUSTER_AGENT_ROOT`。
`start.ipynb` 可按需单独运行后续单元。


In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

configured_root = os.environ.get("CLUSTER_AGENT_ROOT", "").strip()
if configured_root:
    ROOT = Path(configured_root).expanduser().resolve()
    if not (ROOT / "start.ipynb").is_file() or not (ROOT / "scripts").is_dir():
        raise FileNotFoundError(f"CLUSTER_AGENT_ROOT 不是项目根目录: {ROOT}")
else:
    start_dir = Path.cwd().resolve()
    ROOT = next(
        (candidate for candidate in (start_dir, *start_dir.parents)
         if (candidate / "start.ipynb").is_file() and (candidate / "scripts").is_dir()),
        None,
    )
    if ROOT is None:
        raise FileNotFoundError("无法从当前目录向上找到 start.ipynb；请设置 CLUSTER_AGENT_ROOT")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print("CCDC Python API package:", "found" if importlib.util.find_spec("ccdc") else "not found")
print("CCDC license: not checked")
print("Project:", ROOT)

## 1. 筛选 refcode list （根据细致条件再次筛选）

先在 CCDC 中将 search 结果导出为 refcode list（.gcd）。在 Cell 3 配置原始 GCD 路径、筛选后 GCD 的文件名和保存位置；相对路径以项目根目录为基准，也支持绝对路径。Cell 3 默认只读取唯一 refcode 数量和筛选参数；核对输入后将 FILTER_EXECUTE=True 才查询 CSD 并写出筛选 GCD 与报告。

In [ ]:
from scripts.ccdc_data import read_gcd
from scripts.filter_gcd import run as filter_gcd

def project_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else ROOT / path

def resolve_filter_paths(input_path, output_path, report_path=""):
    input_gcd = project_path(input_path).resolve()
    output_gcd = project_path(output_path).resolve()
    report_csv = (
        project_path(report_path).resolve()
        if report_path.strip()
        else output_gcd.with_name(f"{output_gcd.stem}_filter_report.csv")
    )
    if len({input_gcd, output_gcd, report_csv}) != 3:
        raise ValueError("原始 GCD、筛选后 GCD 和筛选报告必须使用不同的文件路径")
    if input_gcd.suffix.lower() != ".gcd" or output_gcd.suffix.lower() != ".gcd":
        raise ValueError("原始列表和筛选后列表的文件扩展名必须为 .gcd")
    if report_csv.suffix.lower() != ".csv":
        raise ValueError("筛选报告的文件扩展名必须为 .csv")
    return input_gcd, output_gcd, report_csv


# 路径可以是项目相对路径或绝对路径。
INPUT_GCD_PATH = "inputs/complex_cluster.gcd"  # 原始 refcode list
FILTERED_GCD_PATH = "data/filtered.gcd"  # 输出文件名和保存位置
FILTER_REPORT_PATH = ""  # 留空时在输出旁生成 <文件名>_filter_report.csv
INPUT_GCD, FILTERED_GCD, FILTER_REPORT = resolve_filter_paths(
    INPUT_GCD_PATH, FILTERED_GCD_PATH, FILTER_REPORT_PATH
)

FILTER_EXECUTE = False  # 预览默认开启；核对条件后改为 True
FILTER_OPTIONS = {
    "must_contain": [],
    "forbid": [],
    "element": [],
    "metals": "",
    "ignore_solvents": False,
    "exclude_formula_n": False,
}

if not INPUT_GCD.is_file():
    raise FileNotFoundError(f"输入 GCD 不存在: {INPUT_GCD}")
input_refcodes = read_gcd(INPUT_GCD)
print(f"输入: {INPUT_GCD}；唯一 refcode: {len(input_refcodes)}")
print(f"筛选结果: {FILTERED_GCD}")
print(f"筛选报告: {FILTER_REPORT}")
print(f"筛选条件: {FILTER_OPTIONS}")
if FILTER_EXECUTE:
    filter_gcd(
        gcd_path=INPUT_GCD,
        out_gcd=FILTERED_GCD,
        report_csv=FILTER_REPORT,
        **FILTER_OPTIONS,
    )
else:
    print(f"预览模式：未查询 CSD 或写入文件。执行后输出: {FILTERED_GCD}；报告: {FILTER_REPORT}")

## 2. 导出结构与 DOI 清单

输入 Cell 3 配置的筛选后 GCD（默认 data/filtered.gcd），生成 data/structures.csv。输出列为 refcode, doi, formula, ccdc_number。Cell 5 默认只核对输入 refcode 数量；将 METADATA_EXECUTE=True 才查询 CSD 并写出 CSV。
cell 5的输入gcd文件位置默认为上一cell保存的位置

In [ ]:
from scripts.ccdc_data import read_gcd
from scripts.export_metadata import run as export_metadata

STRUCTURES_CSV = ROOT / "data" / "structures.csv"

METADATA_EXECUTE = False  # 预览默认开启；核对输入后改为 True
if not FILTERED_GCD.is_file():
    raise FileNotFoundError(f"筛选结果不存在；先执行 Cell 3: {FILTERED_GCD}")
filtered_refcodes = read_gcd(FILTERED_GCD)
print(f"输入: {FILTERED_GCD}；唯一 refcode: {len(filtered_refcodes)}")
if METADATA_EXECUTE:
    export_metadata(gcd_path=FILTERED_GCD, out_csv=STRUCTURES_CSV)
else:
    print(f"预览模式：未查询 CSD 或写入文件。执行后输出: {STRUCTURES_CSV}")

## 3. 导出 CIF

查询方式可选 CCDC 编号或精确 refcode；按编号查询遇多条命中时不导出，按 refcode 查询可处理人工确认的歧义项。Cell 7 默认只读检查输入编号、去重结果和本地已有 CIF，不访问 CSD；因此预览不能判断 CSD 是否有记录、是否多条命中。打开 `CIF_EXPORT_EXECUTE` 后才由独立子进程查询 CSD 并导出，需已激活 CCDC 许可证。已有 CIF 默认跳过；默认使用 `data/process_csv/final_missing_cif_targets.csv` 补缺，全量来源需显式切换。同一 CCDC 编号对应多个 refcode 时分别以编号_refcode 命名。

In [ ]:
import csv
import importlib
from collections import Counter
from scripts import export_cifs as export_cifs_module

importlib.reload(export_cifs_module)
export_cifs = export_cifs_module.run
preview_cif_exports = export_cifs_module.preview

CIF_SOURCE_MODE = "all"  # 新项目先用 structures.csv；"missing" 仅用于cell10审计后补缺
CIF_EXPORT_EXECUTE = False
CIF_EXPORT_OVERWRITE = False
CIF_LOOKUP_BY = "ccdc"
if CIF_SOURCE_MODE == "missing":
    CIF_SOURCE = ROOT / "data" / "process_csv" / "final_missing_cif_targets.csv"
    CIF_LOG = ROOT / "data" / "process_csv" / "export_missing_cifs_log.csv"
elif CIF_SOURCE_MODE == "all":
    CIF_SOURCE = STRUCTURES_CSV
    CIF_LOG = ROOT / "data" / "process_csv" / "export_cifs_log.csv"
else:
    raise ValueError("CIF_SOURCE_MODE 应为 'missing' 或 'all'")

with CIF_SOURCE.open(encoding="utf-8-sig", newline="") as handle:
    cif_targets = list(csv.DictReader(handle))
print(f"CIF 来源: {CIF_SOURCE}；输入行数: {len(cif_targets)}；覆盖已有文件: {CIF_EXPORT_OVERWRITE}")
if CIF_EXPORT_EXECUTE:
    export_cifs(
        csv_path=CIF_SOURCE,
        out_dir=ROOT / "CIFs",
        overwrite=CIF_EXPORT_OVERWRITE,
        log_csv=CIF_LOG,
        lookup_by=CIF_LOOKUP_BY,
        worker_timeout=120,
    )
else:
    preview_rows = preview_cif_exports(
        csv_path=CIF_SOURCE,
        out_dir=ROOT / "CIFs",
        overwrite=CIF_EXPORT_OVERWRITE,
        lookup_by=CIF_LOOKUP_BY,
    )
    print("只读预览状态:", dict(Counter(row["status"] for row in preview_rows)))
    for row in preview_rows:
        print(row["status"], row["ccdc_number"], row["refcode"], row["file"], row["detail"])
    print("只读预览未访问 CSD，也未创建目录或写入日志。设置 CIF_EXPORT_EXECUTE=True 才会导出。")


## 4. 分批下载 PDF

`batch_index` 从 0 开始。先用较小的 `batch_size` 测试；已有的有效 PDF 会跳过。不要轻易更换 DOI 来源 CSV。若中断后修改批大小，请重新检查批次范围，以免遗漏。浏览器可能弹出验证页面，需手动完成；失败 DOI 可从 `failed_dois.csv` 重试。

已有有效 PDF 会跳过，失败的 DOI 可以在相同批次重新运行；失败日志会更新。


In [ ]:
from scripts.download_batches import run as download_batch

DOWNLOAD_BATCH_SIZE = 50
DOWNLOAD_BATCH_INDEX = 0  # 从 0 开始
DOWNLOAD_EXECUTE = False   # 会启动浏览器并访问出版社；确认批次后再改 True

if DOWNLOAD_EXECUTE:
    results = download_batch(
        csv_path=STRUCTURES_CSV,
        pdf_dir=ROOT / "documents" / "pdf",
        batch_size=DOWNLOAD_BATCH_SIZE,
        batch_index=DOWNLOAD_BATCH_INDEX,
        failed_csv=ROOT / "documents" / "failed_dois.csv",
        log_csv=ROOT / "documents" / "download_log.csv",
        challenge_timeout=120,
        headed=True,
    )
else:
    print(f"预览模式：metadata={STRUCTURES_CSV}；batch_index={DOWNLOAD_BATCH_INDEX}；batch_size={DOWNLOAD_BATCH_SIZE}。未启动浏览器。")


## 查看累计下载失败的 DOI
方便后续手动下载

In [ ]:
import csv

failed_path = ROOT / "documents" / "failed_dois.csv"
if failed_path.exists():
    with failed_path.open(encoding="utf-8-sig", newline="") as handle:
        failed = list(csv.DictReader(handle))
    print(f"累计失败 DOI: {len(failed)}")
    for row in failed[:20]:
        print(row["doi"], "-", row["reason"][:120])
else:
    print("尚未生成 failed_dois.csv")


## 5. 使用 MinerU 将 PDF 转成 Markdown

在启动 Jupyter 的终端设置 `MINERU_API_KEY`，不要把 Token 写入 Notebook。转换结果保存到 `documents/md/`。默认仅预览；设置 `MINERU_EXECUTE=True` 后最多转换 5 篇。已有的非空 Markdown 会跳过。扫描件可设置 `is_ocr=True`。

已有非空 Markdown 默认跳过（不启用覆盖），失败文件可重新运行本单元；仅凭非空不代表转换质量合格，请查看 5.1。


In [ ]:
import re
from scripts.mineru_pdf_parser import run_batch

MINERU_EXECUTE = False  # API 会产生用量；核对目录后显式改为 True
MINERU_LIMIT = 5        # 确认结果后可设为 None 全量处理

if MINERU_EXECUTE:
    mineru_results = run_batch(
        pdf_dir=ROOT / "documents" / "pdf", # /"review"
        md_root=ROOT / "documents" / "md",  # /"review" 提取综述的md放在同一目录
        log_csv=ROOT / "data" / "process_csv" / "mineru_conversion_log.csv", #"mineru_review_log.csv"
        limit=MINERU_LIMIT,
        overwrite=False,
        is_ocr=False,
        language="en",
        model_version="vlm",
        poll_seconds=10,
        max_wait=3600,
    )
    success = sum(row["status"] in {"success", "skipped"} for row in mineru_results)
    failed = sum(row["status"] == "failed" for row in mineru_results)
    print(f"本批完成/跳过: {success}；失败: {failed}")
else:
    pdf_count = sum(1 for _ in (ROOT / "documents" / "pdf").glob("*.pdf"))
    md_files = list((ROOT / "documents" / "md").glob("*.md"))
    si_md_count = sum(1 for path in md_files if re.search(r"_si(?:[_-]|$)", path.stem, re.IGNORECASE))
    main_md_count = len(md_files) - si_md_count
    missing_md_count = sum(1 for pdf in (ROOT / "documents" / "pdf").glob("*.pdf")
                           if not (ROOT / "documents" / "md" / f"{pdf.stem}.md").is_file())
    print(f"预览模式：PDF {pdf_count} 篇；主文 Markdown {main_md_count} 篇；SI Markdown {si_md_count} 篇；缺主文 Markdown {missing_md_count} 篇；本次上限 {MINERU_LIMIT}。未调用 MinerU API。")


## 5.1 检查 Markdown 质量

扫描件未开启 OCR 时可能只产生很短的 Markdown。Cell 15 默认只显示 PDF/Markdown 数量；将 QUALITY_EXECUTE=True 才会检查文本长度、写质量报告，并复制可疑文件到隔离目录。

In [ ]:
import re
from scripts.check_markdown_quality import run as check_markdown_quality

QUALITY_EXECUTE = False  # 会写报告并复制问题文件；核对后改为 True
PDF_DIR = ROOT / "documents" / "pdf"
MD_DIR = ROOT / "documents" / "md"
QUALITY_REPORT = ROOT / "data" / "md_quality_issues.csv"
if QUALITY_EXECUTE:
    md_issues = check_markdown_quality(
        pdf_dir=PDF_DIR,
        md_dir=MD_DIR,
        isolate_dir=PDF_DIR / "isolated_tiny_md",
        md_isolate_dir=MD_DIR / "isolated_tiny_md",
        metadata_csv=STRUCTURES_CSV,
        report_csv=QUALITY_REPORT,
        min_md_bytes=10000,
        min_text_chars=2000,
    )
else:
    pdf_count = sum(1 for _ in PDF_DIR.glob("*.pdf"))
    md_files = list(MD_DIR.glob("*.md"))
    si_md_count = sum(1 for path in md_files if re.search(r"_si(?:[_-]|$)", path.stem, re.IGNORECASE))
    main_md_count = len(md_files) - si_md_count
    missing_md_count = sum(1 for pdf in PDF_DIR.glob("*.pdf") if not (MD_DIR / f"{pdf.stem}.md").is_file())
    print(f"预览模式：PDF {pdf_count} 篇；主文 Markdown {main_md_count} 篇；SI Markdown {si_md_count} 篇；缺主文 Markdown {missing_md_count} 篇。未检查、复制或写报告。")
    print(f"执行后报告: {QUALITY_REPORT}；问题文件将复制到各自 isolated_tiny_md 目录。")

## 6. 从文章提取结构卡片

先重建队列，再批量调用 Codex CLI。默认只重建队列并预览；设置 `EXTRACTION_EXECUTE=True` 才调用模型，首次最多 3 篇。该步骤依据 card 与元数据重建队列；无卡、无有效标题或缺少预期 CCDC 的 card 会进入待处理。模型失败可能是超时、DOI 不符等，并非全都需要 SI。合成缺项由第 7 节只读 SI 补丁处理，失败原因见提取日志。

**运行会调用模型并消耗额度。**


In [ ]:
import importlib.util
import sys

EXTRACTION_LIMIT = 3
EXTRACTION_EXECUTE = False  # 会调用 Codex CLI 并写入 card；先核对内存队列
if EXTRACTION_LIMIT is not None and int(EXTRACTION_LIMIT) < 1:
    raise ValueError("EXTRACTION_LIMIT 必须为正整数或 None")

skill_scripts = ROOT / "skills" / "rare-earth-literature-extractor" / "scripts"
queue_path = ROOT / "data" / "process_csv" / "extraction_queue.csv"
build_spec = importlib.util.spec_from_file_location("build_extraction_queue", skill_scripts / "build_queue.py")
build_mod = importlib.util.module_from_spec(build_spec)
build_spec.loader.exec_module(build_mod)
queue_rows = build_mod.build_queue(
    metadata_csv=STRUCTURES_CSV,
    pdf_dir=ROOT / "documents" / "pdf",
    md_dir=ROOT / "documents" / "md",
    si_dir=ROOT / "documents" / "si",
    output_csv=queue_path,
    cards_dir=ROOT / "data" / "extracted_cards",
    missing_csv=ROOT / "data" / "process_csv" / "md_without_ccdc.csv",
    write_files=EXTRACTION_EXECUTE,
)
pending_rows = [row for row in queue_rows if row["status"] != "completed"]
limit = len(pending_rows) if EXTRACTION_LIMIT is None else min(int(EXTRACTION_LIMIT), len(pending_rows))
print(f"待提取: {len(pending_rows)} 篇；本次选择: {limit} 篇；队列目标: {queue_path}", flush=True)
for row in pending_rows[:limit]:
    print(f"  {row['doi']} [{row['source_mode']}]: {row['md_path']}", flush=True)

if EXTRACTION_EXECUTE and pending_rows:
    sys.path.insert(0, str(skill_scripts))
    spec = importlib.util.spec_from_file_location("run_codex_batch", skill_scripts / "run_codex_batch.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    extraction_summary = mod.run(limit=EXTRACTION_LIMIT, use_structure_context=True)
    print(f"最终结果: 成功 {extraction_summary['succeeded']}，失败 {extraction_summary['failed']}", flush=True)
elif not EXTRACTION_EXECUTE:
    print("预览模式：队列仅在内存中生成；未写入 CSV、未调用模型。")
else:
    print("没有待提取文章；没有调用模型。")

### 6.1 根据对账结果整卡重提（可选，非 SI 补全）

先运行 6.3 并将 `CCDC_AUDIT_EXECUTE=True`，生成 `data/process_csv/card_ccdc_audit.csv`。本单元默认从该对账 CSV 选取 `card_identifier_not_in_catalog` 和 `catalog_not_in_card`，且已有 card 的 DOI 整卡重提；缺少 CCDC 标识或缺少 card 不会自动纳入。SI 合成缺口请转第 7 节。重提状态保存在独立的 `card_reextract_queue.csv`，不会改写 6.3 对账报告；成功项默认跳过。


In [ ]:
import csv
import importlib.util
import sys
from scripts.select_reextract_folder import record_result, write_queue

REEXTRACT_SOURCE = "ccdc_audit"  # "ccdc_audit" 或显式选择 "large_cluster"
REEXTRACT_AUDIT_STATUSES = {"card_identifier_not_in_catalog", "catalog_not_in_card"}
REEXTRACT_LIMIT = 3            # None 表示所有待重提 DOI
REEXTRACT_EXECUTE = False      # 预览无模型调用；确认后改 True
REEXTRACT_TIMEOUT = 600

sources = {
    "ccdc_audit": ROOT / "data" / "process_csv" / "card_ccdc_audit.csv",
    "large_cluster": ROOT / "data" / "process_csv" / "large_cluster_papers_gt20.csv",
}
if REEXTRACT_SOURCE not in sources:
    raise ValueError(f"未知清单类型: {REEXTRACT_SOURCE}；请选择 {list(sources)}")
reextract_csv = sources[REEXTRACT_SOURCE]
if REEXTRACT_LIMIT is not None and int(REEXTRACT_LIMIT) < 1:
    raise ValueError("REEXTRACT_LIMIT 必须为正整数或 None")
if not reextract_csv.is_file():
    if REEXTRACT_EXECUTE:
        raise FileNotFoundError(f"先运行 6.3 并保存对账 CSV: {reextract_csv}")
    print(f"尚无候选来源文件: {reextract_csv}；先运行 6.3 并保存对账 CSV。")
    reviewed = []
else:
    with reextract_csv.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if "doi" not in (reader.fieldnames or []):
            raise ValueError(f"清单缺少 doi 列: {reextract_csv}")
        reviewed = list(reader)

queue_path = ROOT / "data" / "process_csv" / "card_reextract_queue.csv"
previous_status = {}
previous_updated = {}
if queue_path.is_file():
    with queue_path.open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            doi = (row.get("doi") or "").strip().lower()
            status = (row.get("reextract_status") or "pending").strip().lower()
            if doi:
                if status not in {"pending", "failed", "success"}:
                    raise ValueError(f"无效的重提状态: {doi}: {status}")
                previous_status[doi] = status
                previous_updated[doi] = row.get("reextract_updated_at") or ""

candidates = {}
for row in reviewed:
    doi = (row.get("doi") or "").strip().lower()
    if not doi:
        continue
    if REEXTRACT_SOURCE == "ccdc_audit":
        issue = (row.get("status") or "").strip()
        if issue not in REEXTRACT_AUDIT_STATUSES:
            continue
        card_path = Path(row.get("card_path") or "")
        if not card_path.is_file():
            continue
        candidate = candidates.setdefault(doi, {"doi": doi, "card_path": str(card_path), "issues": set()})
        candidate["issues"].add(issue)
    else:
        card_value = row.get("card_path") or (ROOT / "data" / "extracted_cards" / f"{doi.replace('/', '_')}.json")
        card_path = project_path(card_value)
        if not card_path.is_file():
            continue
        candidates.setdefault(doi, {"doi": doi, "card_path": str(card_path), "issues": set()})["issues"].add("large_cluster list")

pending = [(doi, item, previous_status.get(doi, "pending"))
           for doi, item in candidates.items() if previous_status.get(doi, "pending") != "success"]
retry_failed = sum(status == "failed" for _, _, status in pending)
limit = len(pending) if REEXTRACT_LIMIT is None else min(int(REEXTRACT_LIMIT), len(pending))
selected = pending[:limit]
print(f"来源: {reextract_csv}；符合条件且已有 card: {len(candidates)}；已成功跳过: {len(candidates) - len(pending)}；失败待重试: {retry_failed}；本次选择: {limit}", flush=True)
for doi, item, status in selected:
    print(f"  [{status}] {doi}: {', '.join(sorted(item['issues']))}", flush=True)

if REEXTRACT_EXECUTE and selected:
    skill_scripts = ROOT / "skills" / "rare-earth-literature-extractor" / "scripts"
    build_spec = importlib.util.spec_from_file_location("build_extraction_queue", skill_scripts / "build_queue.py")
    build_mod = importlib.util.module_from_spec(build_spec)
    build_spec.loader.exec_module(build_mod)
    base_rows = build_mod.build_queue(
        metadata_csv=STRUCTURES_CSV,
        pdf_dir=ROOT / "documents" / "pdf",
        md_dir=ROOT / "documents" / "md",
        si_dir=ROOT / "documents" / "si",
        output_csv=ROOT / "data" / "process_csv" / "extraction_queue.csv",
        cards_dir=ROOT / "data" / "extracted_cards",
        missing_csv=ROOT / "data" / "process_csv" / "md_without_ccdc.csv",
        write_files=False,
    )
    base_by_doi = {(row.get("doi") or "").strip().lower(): row for row in base_rows}
    queue_rows = []
    for doi, item in candidates.items():
        base = base_by_doi.get(doi)
        if not base:
            continue
        queue_rows.append({**base,
                           "audit_issues": "|".join(sorted(item["issues"])),
                           "reextract_status": previous_status.get(doi, "pending"),
                           "reextract_updated_at": previous_updated.get(doi, "")})
    selected_dois = [doi for doi, _, _ in selected if doi in base_by_doi]
    missing_dois = [doi for doi, _, _ in selected if doi not in base_by_doi]
    if missing_dois:
        print("未进入文章队列（缺少可匹配的主文 Markdown/PDF 或元数据）: " + ", ".join(missing_dois), flush=True)
    if selected_dois:
        write_queue(queue_path, queue_rows)
        sys.path.insert(0, str(skill_scripts))
        spec = importlib.util.spec_from_file_location("run_codex_batch", skill_scripts / "run_codex_batch.py")
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        summary = mod.run(
            queue=queue_path, limit=len(selected_dois), dois=selected_dois,
            timeout=REEXTRACT_TIMEOUT, include_completed=True, review_csv=None,
            use_structure_context=True,
            on_result=lambda doi, ok: record_result(queue_path, doi, ok),
        )
        print(f"重提取结果: 成功 {summary['succeeded']}，失败 {summary['failed']}；状态队列: {queue_path}")
    else:
        print("没有候选 DOI 能匹配到主文提取队列。", flush=True)
elif not REEXTRACT_EXECUTE:
    print("预览模式：未调用模型、未写队列或改动 card；先运行并保存 6.3 对账 CSV，再确认后设 REEXTRACT_EXECUTE=True。")
else:
    print("没有可重提文章；没有调用模型。")


### 6.2 按文件夹重提取文章（可选）

把 DOI 命名的文章 `.md` 或 `.pdf` 放入 `reextract-resource/`（可修改路径）；同一篇同时有两种文件时优先 Markdown。文件名须与已有提取队列中的 DOI 文件名一致，未匹配的文件只列出、不重提。先预览，确认后运行会**重建并覆盖整张论文 card**，不会只补结构描述；SI 文件不应放入此目录。
状态直接保存在 `data/process_csv/folder_reextract_queue.csv` 的 `reextract_status` 列：`pending` 未提取、`failed` 下次重试、`success` 默认跳过。先按状态筛选，再应用数量上限；预览不写 CSV。需要强制重提时，把对应行状态改为 `pending`。


In [ ]:
from pathlib import Path
import importlib.util
import sys
from scripts.select_reextract_folder import merge_status, record_result, select_rows, write_queue

FOLDER_REEXTRACT_DIR = ROOT / "reextract-resource"  # DOI 命名的文章 MD/PDF 文件夹
FOLDER_REEXTRACT_LIMIT = 3    # None 表示该文件夹内所有未成功文章
FOLDER_REEXTRACT_EXECUTE = False
FOLDER_REEXTRACT_TIMEOUT = 600

if FOLDER_REEXTRACT_LIMIT is not None and int(FOLDER_REEXTRACT_LIMIT) < 1:
    raise ValueError("FOLDER_REEXTRACT_LIMIT 必须为正整数或 None")

if not FOLDER_REEXTRACT_DIR.is_dir():
    print(f"文件夹不存在: {FOLDER_REEXTRACT_DIR}；先创建文件夹并放入 DOI 命名的文章文件。")
else:
    skill_scripts = ROOT / "skills" / "rare-earth-literature-extractor" / "scripts"
    queue_path = ROOT / "data" / "process_csv" / "extraction_queue.csv"
    folder_queue = ROOT / "data" / "process_csv" / "folder_reextract_queue.csv"
    build_spec = importlib.util.spec_from_file_location("build_extraction_queue", skill_scripts / "build_queue.py")
    build_mod = importlib.util.module_from_spec(build_spec)
    build_spec.loader.exec_module(build_mod)
    queue_rows = build_mod.build_queue(
        metadata_csv=STRUCTURES_CSV,
        pdf_dir=ROOT / "documents" / "pdf",
        md_dir=ROOT / "documents" / "md",
        si_dir=ROOT / "documents" / "si",
        output_csv=queue_path,
        cards_dir=ROOT / "data" / "extracted_cards",
        missing_csv=ROOT / "data" / "process_csv" / "md_without_ccdc.csv",
        write_files=False,
    )
    rows, unmatched = select_rows(FOLDER_REEXTRACT_DIR, queue_rows)
    rows = merge_status(rows, folder_queue)
    pending = [row for row in rows if row["reextract_status"] != "success"]
    limit = len(pending) if FOLDER_REEXTRACT_LIMIT is None else min(int(FOLDER_REEXTRACT_LIMIT), len(pending))
    selected = pending[:limit]
    skipped = len(rows) - len(pending)
    retries = sum(row["reextract_status"] == "failed" for row in pending)
    print(f"匹配 {len(rows)} 篇；已成功跳过 {skipped}，失败待重试 {retries}，未提取 {len(pending) - retries}；本次选择 {limit} 篇；未匹配 {len(unmatched)} 个文件", flush=True)
    for row in selected:
        print(f"  [{row['reextract_status']}] {row['doi']}: {row['md_path'] or row['pdf_path']}", flush=True)
    for name in unmatched[:20]:
        print(f"  未匹配: {name}", flush=True)
    if FOLDER_REEXTRACT_EXECUTE and rows:
        write_queue(folder_queue, rows)
    if FOLDER_REEXTRACT_EXECUTE and selected:
        sys.path.insert(0, str(skill_scripts))
        spec = importlib.util.spec_from_file_location("run_codex_batch", skill_scripts / "run_codex_batch.py")
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        summary = mod.run(queue=folder_queue, limit=limit, timeout=FOLDER_REEXTRACT_TIMEOUT,
                          dois=[row["doi"] for row in selected], include_completed=True, review_csv=None,
                          use_structure_context=True,
                          on_result=lambda doi, ok: record_result(folder_queue, doi, ok))
        print(f"文件夹重提取结果: 成功 {summary['succeeded']}，失败 {summary['failed']}")
    elif not FOLDER_REEXTRACT_EXECUTE:
        print("预览模式：队列由本地文件即时生成且仅在内存中选择；没有改写 CSV 或 card。")
    else:
        print("无待提取文章；没有调用模型。")

### 6.3 文献 card 与结构元数据 CCDC 对账

从 `data/structures.csv` 按 DOI 对比文章 card 内各化合物 CCDC 号与 refcode；默认只预览非匹配项，确认后将 `CCDC_AUDIT_EXECUTE=True` 写入 `data/process_csv/card_ccdc_audit.csv`。该 CSV 是 6.1 默认重提候选来源，因此先运行并保存本节，再运行 6.1。空标识不一定错误，也可能是无晶体沉积；匹配仅说明标识出现在同 DOI 元数据，不证明化合物一一对应。请对照论文、SI、CIF 人工复核，不自动改 card。


In [ ]:
import csv
from collections import Counter
from scripts.audit_card_ccdc import audit, FIELDS

CCDC_AUDIT_EXECUTE = False  # True 写报告；不修改 card
CCDC_AUDIT_REPORT = ROOT / "data" / "process_csv" / "card_ccdc_audit.csv"
ccdc_audit_rows = audit(STRUCTURES_CSV, ROOT / "data" / "extracted_cards")
print("CCDC 对账:", dict(Counter(row["status"] for row in ccdc_audit_rows)))
for row in [r for r in ccdc_audit_rows if r["status"] != "matched"][:20]:
    print(row["status"], row["doi"], row["compound_label"], row["catalog_ccdc_numbers"])
if CCDC_AUDIT_EXECUTE:
    CCDC_AUDIT_REPORT.parent.mkdir(parents=True, exist_ok=True)
    with CCDC_AUDIT_REPORT.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(ccdc_audit_rows)
    print("报告:", CCDC_AUDIT_REPORT)
else:
    print("预览模式：未写 CSV 或 card。")


## 7. 从补充材料补全已有卡片的合成信息

只针对 `data/process_csv/si_required.csv` 标记的 DOI。根据已有卡片及上一轮提取日志中的缺失提示，查看对应 SI 文件，生成小型补丁；**不重读原文、不重建整张卡片**。仅填充缺失的合成字段，保留原有结构及性质信息。SI 未提供确切答案时，条目仍留在待补清单。

- `ONLY_THESE=[]`：自动选择有 SI 的待补 DOI；也可填特定 DOI。
- `SKIP=[]`：排除指定 DOI。
- `RUN_LIMIT=3`：每次最多 3 篇；只有设为 `SI_EXECUTE=True` 才调用模型。
- SI Markdown 命名：`documents/md/<DOI 文件名>_si.md`。如果只有 PDF/DOCX，可放在 `documents/si/`，但建议先转换为 Markdown 以便阅读。

**运行会调用模型并消耗额度。**


In [ ]:
import csv
import importlib.util
from scripts.select_reextract_folder import record_result
import json
import re
import sys

ONLY_THESE = []  # 例如 ["10.1002/anie.201002691"]；留空自动选取
SKIP = []        # 要从自动名单排除的 DOI
RUN_LIMIT = 3     # None 表示全部
SI_EXECUTE = False # 先预览；核对后改为 True
TIMEOUT_SECONDS = 1800

if RUN_LIMIT is not None and int(RUN_LIMIT) < 1:
    raise ValueError("RUN_LIMIT 必须为正整数或 None")

skill_scripts = ROOT / "skills" / "rare-earth-literature-extractor" / "scripts"
def norm(value):
    return (value or "").strip().lower()

def doi_stem(value):
    return re.sub(r"[^\w.\-]+", "_", norm(value), flags=re.ASCII).strip("._")

def read_csv(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

required = read_csv(ROOT / "data" / "process_csv" / "si_required.csv")
si_dir = ROOT / "documents" / "si"
md_dir = ROOT / "documents" / "md"
rows = {}
missing_si = []
for item in required:
    doi = item.get("doi") or ""
    card_value = item.get("card_path") or ROOT / "data" / "extracted_cards" / f"{doi_stem(doi)}.json"
    card_path = project_path(card_value)
    if not card_path.is_file():
        continue
    stem = doi_stem(doi)
    si_paths = sorted({str(p.resolve()) for p in [*md_dir.glob(f"{stem}_si*.md"), *si_dir.glob(f"{stem}_si*")] if p.is_file() and re.search(r"_si(?:[_-]|$)", p.stem, re.IGNORECASE)})
    if not si_paths:
        missing_si.append(doi)
        continue
    rows[norm(doi)] = {
        "doi": doi, "si_paths": json.dumps(si_paths, ensure_ascii=False),
        "card_path": str(card_path.resolve()), "status": "pending",
    }

targets = list(ONLY_THESE) if ONLY_THESE else [item["doi"] for item in required if norm(item.get("doi")) in rows]
skip = {norm(doi) for doi in SKIP}
targets = list(dict.fromkeys(doi for doi in targets if norm(doi) not in skip and norm(doi) in rows))
queue_path = ROOT / "data" / "process_csv" / "si_extraction_queue.csv"
limit = len(targets) if RUN_LIMIT is None else min(int(RUN_LIMIT), len(targets))
selected_targets = targets[:limit]
print(f"待补全且已有 SI: {len(targets)} 篇；本次选择: {limit}；缺少 SI 文件: {len(missing_si)} 篇", flush=True)
if missing_si[:8]:
    print("缺少 SI 示例: " + "; ".join(missing_si[:8]), flush=True)
for doi in targets[:limit]:
    print(f"  {doi}: {len(json.loads(rows[norm(doi)]['si_paths']))} 个 SI 文件", flush=True)

if selected_targets and SI_EXECUTE:
    queue_path.parent.mkdir(parents=True, exist_ok=True)
    with queue_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["doi", "si_paths", "card_path", "status"])
        writer.writeheader()
        writer.writerows(rows[norm(doi)] for doi in selected_targets)
    sys.path.insert(0, str(skill_scripts))
    spec = importlib.util.spec_from_file_location("run_codex_batch", skill_scripts / "run_codex_batch.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    summary = mod.run(queue=queue_path, limit=len(selected_targets), timeout=TIMEOUT_SECONDS, si_completion=True,
                      on_result=lambda doi, ok: record_result(queue_path, doi, ok, status_field="status", updated_field="updated_at"))
    print(f"本批结果: 成功 {summary['succeeded']}，失败 {summary['failed']}", flush=True)
elif selected_targets:
    print(f"预览模式：队列仅在内存中生成；未写入 {queue_path}，未调用模型或改写 card。")
else:
    print("没有可运行的 SI 补全条目。", flush=True)

## 7.1 指定 SI 文件夹补跑（可选）

默认只读取 SI + 已有 card + 上次提取的缺项诊断，生成合成信息补丁，不重读正文、不重建整张 card。需要重新核对全部字段时可开启 `RETRY_FULL_CARD=True`，以 SI 为主要证据生成全新 card；`RETRY_INCLUDE_MAIN=True` 时才附正文 MD/PDF，`RETRY_HINTS_DIR` 可提供与 DOI 文件名同名的 .md/.txt 人工提示。整卡模式成功前会备份旧卡；人工提示和日志是待核查线索，不能覆盖文献证据。默认使用 `data/process_csv/si_required.csv` 中已有 card、指定目录有 SI 的 DOI；也可通过 `RETRY_ONLY_THESE` 显式指定人工审核发现的问题 DOI，即使不在该 CSV 中（仍需旧 card 和 SI Markdown）。不要把普通正文 Markdown 当 SI。


In [ ]:
from pathlib import Path
import csv
import importlib.util
import json
import re
import sys
from scripts.select_reextract_folder import record_result

RETRY_MD_DIR = ROOT / "documents" / "md"  # SI Markdown 默认与主文 Markdown 放在一起
RETRY_ONLY_THESE = []  # 可填 DOI；留空选择该文件夹所有合格文献
RETRY_SKIP = []
RETRY_LIMIT = 3       # None 表示全部
RETRY_EXECUTE = False # 先预览；核对后改 True
RETRY_FULL_CARD = False  # 默认只补丁；True 从 SI 重新生成整张 card
RETRY_INCLUDE_MAIN = False  # 整卡模式可选附带 DOI 正文 MD/PDF
RETRY_HINTS_DIR = ROOT / "documents" / "review_hints"  # 可选 <DOI stem>.md/.txt
RETRY_TIMEOUT_SECONDS = 1800

if RETRY_LIMIT is not None and int(RETRY_LIMIT) < 1:
    raise ValueError("RETRY_LIMIT 必须为正整数或 None")

def retry_norm(value):
    return (value or "").strip().lower()

def retry_stem(doi):
    return re.sub(r"[^\w.\-]+", "_", retry_norm(doi), flags=re.ASCII).strip("._")

if not RETRY_MD_DIR.is_dir():
    if RETRY_EXECUTE:
        raise FileNotFoundError(f"SI Markdown folder missing: {RETRY_MD_DIR}")
    print(f"SI Markdown folder not yet present: {RETRY_MD_DIR}")
retry_required_path = ROOT / "data" / "process_csv" / "si_required.csv"
if retry_required_path.is_file():
    with retry_required_path.open(encoding="utf-8-sig", newline="") as handle:
        retry_required = list(csv.DictReader(handle))
else:
    retry_required = []
retry_files = sorted(p for p in RETRY_MD_DIR.glob("*.md") if p.is_file() and p.stat().st_size > 0)
retry_si_file_count = sum(1 for p in retry_files if re.search(r"_si(?:[_-]|$)", p.stem, re.IGNORECASE))
retry_only = {retry_norm(doi) for doi in RETRY_ONLY_THESE}
known_required = {retry_norm(row.get("doi")) for row in retry_required}
retry_required.extend({"doi": doi} for doi in sorted(retry_only - known_required))
retry_skip = {retry_norm(doi) for doi in RETRY_SKIP}
retry_rows = []
for item in retry_required:
    doi = item.get("doi") or ""
    if not doi or retry_norm(doi) in retry_skip or (retry_only and retry_norm(doi) not in retry_only):
        continue
    stem = retry_stem(doi) + "_si"
    matching = [p for p in retry_files if re.search(r"^" + re.escape(stem) + r"(?:[_-]|$)", p.stem, re.IGNORECASE)]
    if not matching:
        continue
    card_value = item.get("card_path") or ROOT / "data" / "extracted_cards" / f"{retry_stem(doi)}.json"
    card_path = project_path(card_value)
    if not card_path.is_file():
        print(f"跳过 {doi}: card 不存在 ({card_path})")
        continue
    hint_path = next((p for ext in (".md", ".txt") if (p := RETRY_HINTS_DIR / (retry_stem(doi) + ext)).is_file()), None)
    retry_rows.append({
        "doi": doi,
        "si_paths": json.dumps([str(p.resolve()) for p in matching], ensure_ascii=False),
        "card_path": str(card_path.resolve()),
        "status": "pending",
        "hint_path": str(hint_path.resolve()) if RETRY_FULL_CARD and hint_path else "",
        "md_path": str((ROOT / "documents" / "md" / (retry_stem(doi) + ".md")).resolve()) if RETRY_FULL_CARD and RETRY_INCLUDE_MAIN and (ROOT / "documents" / "md" / (retry_stem(doi) + ".md")).is_file() else "",
        "pdf_path": str((ROOT / "documents" / "pdf" / (retry_stem(doi) + ".pdf")).resolve()) if RETRY_FULL_CARD and RETRY_INCLUDE_MAIN and (ROOT / "documents" / "pdf" / (retry_stem(doi) + ".pdf")).is_file() else "",
    })

retry_queue = ROOT / "data" / "process_csv" / "si_reextract_queue.csv"
retry_limit = len(retry_rows) if RETRY_LIMIT is None else min(int(RETRY_LIMIT), len(retry_rows))
retry_selected = retry_rows[:retry_limit]
print(f"选定文件夹: {RETRY_MD_DIR}; 非空 Markdown: {len(retry_files)}; SI Markdown: {retry_si_file_count}; 待重试 DOI: {len(retry_rows)}; 本次: {retry_limit}")
for row in retry_selected:
    print(f"  {row['doi']} -> {', '.join(json.loads(row['si_paths']))}")
if retry_only:
    not_selected = retry_only - {retry_norm(row['doi']) for row in retry_rows}
    if not_selected:
        print("未匹配到可补 DOI: " + ", ".join(sorted(not_selected)))

if RETRY_EXECUTE and retry_selected:
    retry_queue.parent.mkdir(parents=True, exist_ok=True)
    if RETRY_FULL_CARD:
        with STRUCTURES_CSV.open(encoding="utf-8-sig", newline="") as handle:
            catalog = list(csv.DictReader(handle))
        for row in retry_selected:
            matched = [item for item in catalog if (item.get("doi") or "").strip().lower() == retry_norm(row["doi"])]
            row["metadata_rows"] = json.dumps(matched, ensure_ascii=False)
    with retry_queue.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["doi", "si_paths", "card_path", "status", "hint_path", "md_path", "pdf_path", "metadata_rows"])
        writer.writeheader()
        writer.writerows({**row, "metadata_rows": row.get("metadata_rows", "[]")} for row in retry_selected)
    retry_scripts = ROOT / "skills" / "rare-earth-literature-extractor" / "scripts"
    sys.path.insert(0, str(retry_scripts))
    retry_spec = importlib.util.spec_from_file_location("run_codex_batch", retry_scripts / "run_codex_batch.py")
    retry_mod = importlib.util.module_from_spec(retry_spec)
    retry_spec.loader.exec_module(retry_mod)
    retry_summary = retry_mod.run(queue=retry_queue, limit=len(retry_selected), timeout=RETRY_TIMEOUT_SECONDS, si_completion=not RETRY_FULL_CARD, si_full_reextract=RETRY_FULL_CARD, review_csv=None,
                                 on_result=lambda doi, ok: record_result(retry_queue, doi, ok, status_field="status", updated_field="updated_at"))
    print(f"本批结果: 成功 {retry_summary['succeeded']}，失败 {retry_summary['failed']}")
elif not RETRY_EXECUTE:
    print(f"预览模式：未写入 {retry_queue}，未调用模型或改动 card。")
else:
    print("没有可重试的条目。")


## 8. 从综述 Markdown 提取知识卡片

综述与原始研究论文的结构卡片分开处理：**一个可复用的知识点一张卡片**，如通用合成策略、结构规律、构效关系。模型需通读整篇 Markdown；输出到 `data/review_knowledge_cards/`，并更新 `data/process_csv/review_knowledge_index.csv`。不修改现有化合物 card 或 SI 流程。

把综述 Markdown 放在同一个文件夹里（默认 `documents/review/`）。文件名沿用本项目的 DOI 规则：只把 DOI 中的第一个 `/` 换成 `_`，例如 `10.1016/j.ccr.2017.10.023` → `10.1016_j.ccr.2017.10.023.md`。同目录中的 PDF 会忽略；空文件、重名 DOI，或无法按该规则还原的文件名会跳过并列出。

默认 `REVIEW_EXECUTE=False` 只预览文件夹里的名单；确认后改成 `True`，每批最多处理 `REVIEW_LIMIT=3` 篇。已在索引里的 DOI 自动跳过；重新提取需显式设置 `REVIEW_REPLACE=True` 并先检查原有卡片。

**实际运行会调用 Codex CLI，消耗模型额度。**


In [ ]:
from pathlib import Path
import importlib.util
import re

REVIEW_DIR = ROOT / "documents" / "review"  # 综述 Markdown 放在一起的文件夹
REVIEW_LIMIT = 3        # 本次最多处理的综述篇数
REVIEW_EXECUTE = False  # 先预览；确认后改为 True
REVIEW_REPLACE = False   # 默认跳过已提取 DOI；重提取前先审查旧卡片
REVIEW_TIMEOUT = 360    # 单篇超时秒数

def doi_from_review_filename(path: Path) -> str:
    """按项目文件名规则还原 DOI：只把第一个 / 写成了 _。"""
    prefix, sep, rest = path.stem.strip().partition("_")
    if not sep or not prefix.lower().startswith("10."):
        raise ValueError(f"文件名无法还原 DOI：{path.name}")
    return f"{prefix}/{rest}"

if not REVIEW_DIR.is_dir():
    raise FileNotFoundError(f"综述文件夹不存在：{REVIEW_DIR}")

review_rows = []
skipped = []
seen = set()
for md_path in sorted(REVIEW_DIR.glob("*.md"), key=lambda item: item.name.lower()):
    if not md_path.is_file() or md_path.stat().st_size == 0:
        skipped.append(f"{md_path.name}（空文件）")
        continue
    try:
        doi = doi_from_review_filename(md_path)
    except ValueError as exc:
        skipped.append(str(exc))
        continue
    if not re.fullmatch(r"10\.\d{4,9}/\S+", doi, flags=re.IGNORECASE):
        skipped.append(f"{md_path.name}（不是合法 DOI）")
        continue
    key = doi.lower()
    if key in seen:
        skipped.append(f"{md_path.name}（DOI 重复：{doi}）")
        continue
    seen.add(key)
    review_rows.append({"doi": doi, "md_file": str(md_path.resolve()), "title": ""})

print(f"文件夹 {REVIEW_DIR}：可用综述 {len(review_rows)} 篇")
for row in review_rows:
    print(f"  {row['doi']} -> {row['md_file']}")
if skipped:
    print("跳过：")
    for item in skipped:
        print(f"  {item}")

review_script = ROOT / "skills" / "review-knowledge-extractor" / "scripts" / "run_review_batch.py"
review_spec = importlib.util.spec_from_file_location("run_review_batch", review_script)
review_module = importlib.util.module_from_spec(review_spec)
review_spec.loader.exec_module(review_module)
review_summary = review_module.run(
    review_rows,
    limit=REVIEW_LIMIT,
    execute=REVIEW_EXECUTE,
    replace=REVIEW_REPLACE,
    timeout=REVIEW_TIMEOUT,
)
print(f"综述本批：选中 {review_summary['total']}；成功 {review_summary['succeeded']}；失败 {review_summary['failed']}")


## 8.1 生成大核团簇 DOI 名单（晶体结构卡片前）

从现有文章 card 筛选金属核数 >20 且有本地 PDF 的 DOI；执行后重建 `data/process_csv/large_cluster_papers_gt20.csv`，并保留已有重提状态。仅预览不会写文件。随后第 9 节可用这份名单限定 CIF 卡片范围（用于在card内记录结构基元）。


In [ ]:
import csv
import json
import subprocess
import sys

LARGE_LIST_EXECUTE = False
LARGE_LIST_PATH = ROOT / "data" / "process_csv" / "large_cluster_papers_gt20.csv"
large_candidates = []
for card_file in sorted((ROOT / "data" / "extracted_cards").glob("*.json")):
    pdf = ROOT / "documents" / "pdf" / (card_file.stem + ".pdf")
    if not pdf.is_file():
        continue
    card = json.loads(card_file.read_text(encoding="utf-8-sig"))
    qualifying = [item for item in card.get("list_of_rare_earth_compounds", [])
                  if isinstance((item.get("material_class") or {}).get("nuclearity"), int)
                  and not isinstance((item.get("material_class") or {}).get("nuclearity"), bool)
                  and (item.get("material_class") or {}).get("nuclearity") > 20]
    if qualifying:
        large_candidates.append((card.get("doi"), max(item["material_class"]["nuclearity"] for item in qualifying)))
print(f"大核候选文献 {len(large_candidates)} 篇；名单: {LARGE_LIST_PATH}")
for doi, nuclearity in sorted(large_candidates, key=lambda item: -item[1])[:15]:
    print(doi, nuclearity)
if LARGE_LIST_EXECUTE:
    subprocess.run([sys.executable, str(ROOT / "scripts" / "select_large_cluster_pdfs.py")], cwd=ROOT, check=True)
else:
    print("预览模式：未写名单。第 9 节 large_cluster 范围需先生成名单。")


## 9. 从 CIF 提取晶体结构卡片

以 CIF 的 data_ 块 refcode 为主，在 `data/cif_structure_cards/` 生成独立 JSON 结构卡片；多数据块按块内 CCDC 沉积号定位，少数旧 CIF 使用已核对的块名映射。若块名不是 refcode，则用论文 card 中该 CCDC 编号唯一对应的 refcode；没有 refcode 时使用 `CCDC_编号`，并记录 `source_data_block`。当前 metadata CSV 只用于 DOI 关联，不是 CIF 总账；`CIF_CARD_SCOPE="large_cluster"` 按大核论文 card 中 >20 核化合物编号筛选，`"all"` 才全量处理。

下方代码单元默认用 `CIF_CARD_EXECUTE=False` 扫描并列出最多 3 个候选，不写结构卡、不重建高质量样本集。确认目标后设为 `True` 执行；已有结构卡默认跳过。重提后刷新高质量样本集；如需更新全局核对总账，另运行 `scripts/audit_integrity.py`。

通过 CIF 键表中共享的 O 标签可以识别桥接金属对。metal_framework.links 记录了金属-金属间距（不是直接键）、每个金属-氧距离以及金属-氧-金属角；connected_components、topology 和 approximate_shape 总结了金属图。Geometry 使用分数坐标和晶胞参数以及周期性图像。只包含同一性对称键，因此省略的对称键以及不明确或缺失的坐标需要人工检查。

In [ ]:
from collections import Counter
import csv
import json
import sys
import importlib
import scripts.extract_cif_structures as cif_extraction
importlib.reload(cif_extraction)
extract_cif_cards = cif_extraction.run

CIF_CARD_SCOPE = "large_cluster"  # "large_cluster" 或 "all"
CIF_CARD_LIMIT = 3  # 每次预览/执行的结构数；确认后可设为 None
CIF_CARD_EXECUTE = False  # 预览默认开启；确认目标后改为 True
CIF_CARD_OVERWRITE = False  # 仅在有意刷新旧卡片时改为 True
if CIF_CARD_SCOPE == "large_cluster":
    with (ROOT / "data" / "process_csv" / "large_cluster_papers_gt20.csv").open(encoding="utf-8-sig", newline="") as handle:
        papers = list(csv.DictReader(handle))
    CIF_REFCODES, CIF_CCDC_NUMBERS = set(), set()
    for paper in papers:
        card = json.loads((ROOT / paper["card_path"]).read_text(encoding="utf-8"))
        for compound in card.get("list_of_rare_earth_compounds", []):
            nuclearity = (compound.get("material_class") or {}).get("nuclearity")
            if isinstance(nuclearity, int) and not isinstance(nuclearity, bool) and nuclearity > 20:
                CIF_REFCODES.update(str(value).upper() for value in compound.get("ccdc_refcodes") or [] if value)
                CIF_CCDC_NUMBERS.update(str(value) for value in compound.get("ccdc_numbers") or [] if value)
    print(f"大核名单：{len(papers)} 篇；目标 refcode {len(CIF_REFCODES)} 个，CCDC 号 {len(CIF_CCDC_NUMBERS)} 个", flush=True)
elif CIF_CARD_SCOPE == "all":
    CIF_REFCODES, CIF_CCDC_NUMBERS = None, None
else:
    raise ValueError("CIF_CARD_SCOPE 应为 large_cluster 或 all")

def cif_progress(stage, current, total, filename):
    action = "扫描 CIF" if stage == "scan" else "处理结构卡"
    suffix = f"：{filename}" if filename else ""
    print(f"{action} {current}/{total}{suffix}", flush=True)

results = extract_cif_cards(
    cif_dir=ROOT / "CIFs",
    metadata=STRUCTURES_CSV,
    out_dir=ROOT / "data" / "cif_structure_cards",
    limit=CIF_CARD_LIMIT,
    overwrite=CIF_CARD_OVERWRITE,
    refcodes=CIF_REFCODES,
    ccdc_numbers=CIF_CCDC_NUMBERS,
    progress=cif_progress,
    preview=not CIF_CARD_EXECUTE,
)
print("处理状态", dict(Counter(row["status"] for row in results)))
for row in results:
    if row["status"] in {"preview", "would_skip"}:
        print(row["status"], row.get("refcode", ""), row.get("ccdc_number", ""), row["file"], row["detail"])
    elif row["status"] in {"unmatched", "failed"}:
        print(row["status"], row["file"], row["detail"])
if not CIF_CARD_EXECUTE:
    print("预览模式：没有写入结构卡，也未重建高质量样本集。设置 CIF_CARD_EXECUTE=True 后才会执行。")

## 10. 生成文件完整性与 CCDC 对账总账

审核 PDF/SI/Markdown/文献 card/CIF 与 `data/structures.csv` 的对应关系。执行会覆盖生成 `data/final_integrity_ledger.csv`、`data/final_integrity_issues.csv`、`data/final_synthesis_gaps.csv`、`data/process_csv/final_missing_cif_targets.csv`，并更新 `data/card_review.md` 的自动审计段；不会修改源 card。


In [ ]:
import subprocess
import sys

INTEGRITY_EXECUTE = False  # 核对前置输入后改 True；会覆盖总账及审计报告
print("Metadata:", STRUCTURES_CSV, "存在:", STRUCTURES_CSV.is_file())
print("执行后生成: data/final_integrity_ledger.csv, data/final_integrity_issues.csv 等")
if INTEGRITY_EXECUTE:
    subprocess.run([sys.executable, str(ROOT / "scripts" / "audit_integrity.py"),
                    "--metadata", str(STRUCTURES_CSV)], cwd=ROOT, check=True)
else:
    print("预览模式：未写入总账或审计报告。")
